# Prepare input files for the encoding pipeline

This includes processing annotated XML files of Trevoux 1743 (Le Robert version) to remove certain tags and clean up the text for LLM input.

 

In [16]:
import os
import re
#import html

In [ ]:
def remove_tags(texte):
    # 1. Supprimer la balise <Nat_Art> ET tout son contenu.
    # Le flag re.DOTALL assure que ça fonctionne même si le contenu s'étale sur plusieurs lignes.
    texte = re.sub(r'<Nat_Art[^>]*>.*?</Nat_Art>', '', texte, flags=re.IGNORECASE | re.DOTALL)
    
    # 2. Remplacer la balise <article> par un saut de ligne
    texte = re.sub(r'<article[^>]*>', '\n', texte, flags=re.IGNORECASE)
    
    # 3. Supprimer toutes les autres balises (</article>, <page>, <G>, <vedette>, etc.).
    texte = re.sub(r'<[^>]+>', '', texte)
    
    # 4. Décoder les entités HTML (&amp; devient &).
    #texte = html.unescape(texte)
    
    # 5. Nettoyer les sauts de lignes vides.
    # Cette regex cible 2 sauts de lignes (ou plus) séparés par des espaces ou des tabulations,
    # et les réduit à un seul saut de ligne. Cela élimine les lignes vides sans casser le texte.
    texte = re.sub(r'\n[ \t\n]*\n', '\n', texte)

    # supprime les tabulations résiduels
    texte = re.sub(r'[\t]+', '', texte)
    
    # 6. Supprimer les espaces et sauts de ligne résiduels au tout début et à la toute fin du texte.
    return texte.strip()


In [18]:
INPUT_PATH = os.path.join("data", "robert", "pages")
OUTPUT_PATH = os.path.join("data", "robert", "pages-txt")

for filename in os.listdir(INPUT_PATH):
    if filename.endswith(".xml"):
         with open(os.path.join(INPUT_PATH, filename), "r", encoding="utf-8") as f:
            raw_txt = f.read()
            cleaned_txt = remove_tags(raw_txt)
            print(f"--- Contenu nettoyé de {filename} ---")
            print(cleaned_txt)
            print("\n\n")

            # saver le texte nettoyé dans un nouveau fichier .txt
            output_filename = filename.replace(".xml", ".txt")
            with open(os.path.join(OUTPUT_PATH, output_filename), "w", encoding="utf-8") as out_f:
                out_f.write(cleaned_txt)    


--- Contenu nettoyé de TR5_p489-490.xml ---
PRÉSERVER. v. act. Garder, garantir de quelque mal. Servare, custodire, tueri, defendere. On dit par forme de souhait, Dieu vous préserve de mal &amp; de fortune. Une bonne cuirasse l'a préservé de plusieurs mousquetades. Le zèle de cet homme qui a secouru les pestiférés, l'a préservé jusqu'ici. Une saignée faite à propos préserve d'une maladie qui menaçoit. Le sel préserve de corruption. Aimes la raison, &amp; elle te préservera de la contagion du mauvais exemple. M. Esp.
Préservé, ée. part. pass. &amp; adj. Servatus, tutus.
PRÉSIDENCE. subst. f. La qualité de Président. Praesidis dignitas. La première Présidence d'un tel Parlement est vacante. Il y a force brigue pour cette Présidence.
PRÉSIDENT. s. m. Chef, ou Modérateur d'une Compagnie, d'une Assemblée. Praeses, Moderator. Le Président de l'Assemblée du Clergé, le Président des États. Le plus ancien, le Doyen est d'ordinaire le Président, où il n'y en a point de créé, ou de présent.
Prési

In [5]:
# --- Test  ---
contenu_brut = """<page>présume devoir hériter, s'il n'en est point empêché par la disposition contraire d'un testateur. <I>Praesumptivus.</I>
</article>
<article ID="250401890">
<Nat_Art>Trévoux 1743</Nat_Art>
	<G><vedette>PRÉSOMPTION.</vedette></G> <cat_gra>s. f.</cat_gra> Orgueil : trop bonne opinion qu'on a de soi-même, &amp; qui fait traiter les autres avec mépris. 
<I>Praesumtio, arrogantia.</I> Les Auteurs sont sujets à avoir une sotte <I>présomption,</I> une ridicule vanité.
</article>
<article ID="250401900">
<Nat_Art>Trévoux 1743</Nat_Art>
	<G><vedette>PRÉSOMPTUEUSEMENT.</vedette></G> <cat_gra>adv.</cat_gra> D'une manière présomptueuse.
</article></page>"""

texte_propre = remove_tags(contenu_brut)
print(texte_propre)

présume devoir hériter, s'il n'en est point empêché par la disposition contraire d'un testateur. Praesumptivus.
	PRÉSOMPTION. s. f. Orgueil : trop bonne opinion qu'on a de soi-même, &amp; qui fait traiter les autres avec mépris. 
Praesumtio, arrogantia. Les Auteurs sont sujets à avoir une sotte présomption, une ridicule vanité.
	PRÉSOMPTUEUSEMENT. adv. D'une manière présomptueuse.
